In [ ]:
!nvidia-smi

Mon Sep 14 17:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!uname -a

Linux ada488137dac 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


In [ ]:
%%writefile info.cu
#include <stdio.h>

int main() {
	int deviceCount;
	cudaDeviceProp devProp;
	cudaGetDeviceCount(&deviceCount);
	printf("Found %d devices\n", deviceCount);
	for(int device = 0;device < deviceCount;device++) {
		cudaGetDeviceProperties(&devProp, device);
		printf("Device %d\n", device);
		printf("Compute capability      : %d.%d\n", devProp.major, devProp.minor);
		printf("Name                    : %s\n", devProp.name);
		printf("Total Global Memory     : %zu\n", devProp.totalGlobalMem);
		printf("Shared memory per block : %zu\n", devProp.sharedMemPerBlock);
		printf("Registers per block     : %d\n", devProp.regsPerBlock);
		printf("Warp size               : %d\n", devProp.warpSize);
		printf("Max threads per block   : (%d, %d, %d)\n", devProp.maxThreadsDim[0], devProp.maxThreadsDim[1], devProp.maxThreadsDim[2]);
		printf("Max block   : (%d, %d, %d)\n", devProp.maxGridSize[0], devProp.maxGridSize[1], devProp.maxGridSize[2]);
		printf("Total constant memory   : %zu\n", devProp.totalConstMem);
		printf("Multiprocessors count   : %d\n", devProp.multiProcessorCount);
	}
	return 0;
}

Writing info.cu


In [ ]:
%%shell
nvcc -arch=sm_75 info.cu -o info
./info

Found 1 devices
Device 0
Compute capability      : 7.5
Name                    : Tesla T4
Total Global Memory     : 15637086208
Shared memory per block : 49152
Registers per block     : 65536
Warp size               : 32
Max threads per block   : (1024, 1024, 64)
Max block   : (2147483647, 65535, 65535)
Total constant memory   : 65536
Multiprocessors count   : 40


In [5]:
%%writefile array.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void kernel(float *arr, int n) {
/*
    threadIdx.x
    blockIdx.x
    blockDim.x
    gridDim.x
*/
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    printf("threadIdx.x=%d, blockIdx.x=%d, idx=%d\n", threadIdx.x, blockIdx.x, idx);
    while(idx < n) {
        arr[idx] *= 2.0;
        idx += offset;
    }
}

int main() {
    int i, n = 10000000;
    float *arr = (float *)malloc(sizeof(float) * n);
    for(i = 0; i < n; i++)
        arr[i] = i;
    float *dev_arr;
    cudaMalloc(&dev_arr, sizeof(float) * n);
    cudaMemcpy(dev_arr, arr, sizeof(float) * n, cudaMemcpyHostToDevice);

    kernel<<<4, 4>>>(dev_arr, n);
    cudaDeviceSynchronize();
    printf("Hello World!\n");
    cudaMemcpy(arr, dev_arr, sizeof(float) * n, cudaMemcpyDeviceToHost);
    for(i = 0; i < 10; i++)
        printf("%f ", arr[i]);
    printf("\n");

    cudaFree(dev_arr);
    free(arr);
    return 0;
}

Overwriting array.cu


In [6]:
%%shell
nvcc -arch=sm_75 array.cu -o array
./array

threadIdx.x=0, blockIdx.x=2, idx=8
threadIdx.x=1, blockIdx.x=2, idx=9
threadIdx.x=2, blockIdx.x=2, idx=10
threadIdx.x=3, blockIdx.x=2, idx=11
threadIdx.x=0, blockIdx.x=0, idx=0
threadIdx.x=1, blockIdx.x=0, idx=1
threadIdx.x=2, blockIdx.x=0, idx=2
threadIdx.x=3, blockIdx.x=0, idx=3
threadIdx.x=0, blockIdx.x=3, idx=12
threadIdx.x=1, blockIdx.x=3, idx=13
threadIdx.x=2, blockIdx.x=3, idx=14
threadIdx.x=3, blockIdx.x=3, idx=15
threadIdx.x=0, blockIdx.x=1, idx=4
threadIdx.x=1, blockIdx.x=1, idx=5
threadIdx.x=2, blockIdx.x=1, idx=6
threadIdx.x=3, blockIdx.x=1, idx=7
Hello World!
0.000000 2.000000 4.000000 6.000000 8.000000 10.000000 12.000000 14.000000 16.000000 18.000000 


In [11]:
%%writefile csc_and_time.cu
#include <stdio.h>
#include <stdlib.h>

#define CSC(call)  									                \
do {											                    \
	cudaError_t res = call;							                \
	if (res != cudaSuccess) {							            \
		fprintf(stderr, "ERROR in %s:%d. Message: %s\n",			\
				__FILE__, __LINE__, cudaGetErrorString(res));		\
		exit(0);								                    \
	}										                        \
} while(0)

__global__ void kernel(int *arr, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    while (idx < n) {
        arr[idx] *= 2;
        idx += offset;
    }
}

int main() {
    int i, n = 100000000;
    int *arr = (int *)malloc(sizeof(int) * n);
    for(i = 0; i < n; i++)
        arr[i] = i;
    int *dev_arr;
    CSC(cudaMalloc(&dev_arr, sizeof(int) * n));
    CSC(cudaMemcpy(dev_arr, arr, sizeof(int) * n, cudaMemcpyHostToDevice));

    cudaEvent_t start, stop;
    CSC(cudaEventCreate(&start));
    CSC(cudaEventCreate(&stop));

    CSC(cudaEventRecord(start));

    kernel<<< 1024, 1024 >>>(dev_arr, n);

    CSC(cudaEventRecord(stop));

 //   CSC(cudaDeviceSynchronize());

    CSC(cudaEventSynchronize(stop));
    CSC(cudaGetLastError());

    float t;
    CSC(cudaEventElapsedTime(&t, start, stop));
    CSC(cudaEventDestroy(start));
    CSC(cudaEventDestroy(stop));

    printf("time = %f ms\n", t);

    CSC(cudaMemcpy(arr, dev_arr, sizeof(int) * n, cudaMemcpyDeviceToHost));
    for(i = 0; i < 10; i++)
        printf("%d ", arr[i]);
    printf("\n");
    CSC(cudaFree(dev_arr));
    free(arr);
    return 0;
}

Overwriting csc_and_time.cu


In [12]:
%%shell
nvcc -arch=sm_75 csc_and_time.cu -o csc_and_time
./csc_and_time

ERROR in csc_and_time.cu:45. Message: invalid configuration argument


In [ ]:
!ncu --query-metrics

Device Tesla T4 (TU104)
--------------------------------------------------------------------------- --------------- --------------- ----------------------------------------------------------------------
Metric Name                                                                 Metric Type     Metric Unit     Metric Description                                                    
--------------------------------------------------------------------------- --------------- --------------- ----------------------------------------------------------------------
dram__bytes                                                                 Counter         byte            # of bytes accessed in DRAM                                           
dram__bytes_read                                                            Counter         byte            # of bytes read from DRAM                                             
dram__bytes_write                                                           Count